# PEA Met Network Analysis

OSEMN pipeline analysis for Parks Canada PEI Field Unit weather station data.

This notebook answers four core questions:
1. **Are any PCA stations redundant?**
2. **How similar are PCA stations to the ECCC Stanhope reference?**
3. **Can daily FWI values be calculated reliably for Cavendish and Greenwich?**
4. **What is the uncertainty of removing a station?**

Sections:
1. **Obtain & Scrub** — data loading, QA/QC, imputation
2. **Explore** — EDA and summary statistics
3. **Model** — FWI calculation, redundancy analysis, uncertainty
4. **Interpret** — findings and recommendations

In [ ]:
import sys
sys.path.insert(0, "src")

import pandas as pd
import numpy as np
from pathlib import Path

print("PEA Met Network Analysis")
print("OSEMN Pipeline — Parks Canada PEI Field Unit")
print()

## 1. Obtain & Scrub

Loading cleaned, processed data from `data/processed/`. The pipeline
(in `cleaning.py`) has already performed normalization, imputation,
QA/QC, and FWI computation. Here we load the results.

In [ ]:
from pathlib import Path

processed_dir = Path("data/processed")

# Discover PCA stations (exclude stanhope reference and sample data)
pca_stations = sorted([
    d.name for d in processed_dir.iterdir()
    if d.is_dir()
    and d.name not in ("stanhope", "cavendish_dec2022_sample")
])

print(f"PCA stations found: {pca_stations}")
print(f"Total: {len(pca_stations)} stations\n")

# Load processed data for each station
station_data = {}
for station in pca_stations:
    sdir = processed_dir / station
    hourly_path = sdir / "station_hourly.csv"
    daily_path = sdir / "station_daily.csv"
    
    if hourly_path.exists():
        station_data[station] = {
            "hourly": pd.read_csv(hourly_path, parse_dates=["timestamp_utc"]),
        }
    if daily_path.exists():
        station_data[station]["daily"] = pd.read_csv(
            daily_path, parse_dates=["timestamp_utc"]
        )

# Load Stanhope reference
stanhope_hourly = pd.read_csv(
    processed_dir / "stanhope" / "station_hourly.csv",
    parse_dates=["timestamp_utc"],
)
station_data["stanhope"] = {"hourly": stanhope_hourly}

# Load imputation report
imputation_report = pd.read_csv(processed_dir / "imputation_report.csv")

# Summary table
print("=== Station Data Summary ===")
for name, data in station_data.items():
    h_rows = len(data["hourly"])
    d_rows = len(data.get("daily", pd.DataFrame()))
    h_cols = list(data["hourly"].columns)
    print(f"  {name}: {h_rows} hourly rows, {d_rows} daily rows")
    print(f"    Hourly columns: {h_cols}")

print(f"\nImputation report: {len(imputation_report)} records")

## 2. Explore — EDA

Exploratory data analysis covers temporal coverage, missingness patterns,
and variable distributions across all stations.

In [ ]:
from pea_met_network.qa_qc import missingness_summary

# EDA placeholder — will be expanded in subsequent loops
print("QA/QC summary functions available.")
print("See notebooks/01_explore.ipynb for full EDA.")

## 3. Model

### FWI Calculation
Full FWI chain: FFMC → DMC → DC → ISI → BUI → FWI
implemented in `src/pea_met_network/fwi.py` with Van Wagner formulas.

### Redundancy Analysis
PCA-based station redundancy analysis in `src/pea_met_network/redundancy.py`.

### Uncertainty Quantification
Probabilistic uncertainty in `src/pea_met_network/uncertainty.py`.

In [ ]:
from pea_met_network import fwi, redundancy, uncertainty
print("FWI, redundancy, and uncertainty modules loaded.")

## 4. Interpret

Key findings and recommendations will be documented here
as the analysis progresses.